In [2]:
from pathlib import Path
import duckdb
import pandas as pd

class DataHubReader:
    def __init__(self, db_name="yahoo_finance.db"):
        """
        Stellt Verbindung zur DuckDB her.
        Erwartet dieselbe Ordnerstruktur wie DataHub.
        """
        base_path = Path.cwd()
        db_path = base_path.parent.parent / "data" / "01_raw" / "yahoo" / db_name

        if not db_path.exists():
            raise FileNotFoundError(f"Datenbank nicht gefunden: {db_path}")

        self.con = duckdb.connect(str(db_path))

    def _fetch_table(self, table_name: str) -> pd.DataFrame:
        """Generische Methode zum Laden einer Tabelle."""
        return self.con.execute(f"SELECT * FROM {table_name}").df()

    def get_financials(self) -> pd.DataFrame:
        """Yahoo Finance Financials"""
        return self._fetch_table("bronze_financials")

    def get_wikidata(self) -> pd.DataFrame:
        """Wikidata Unternehmensdaten"""
        return self._fetch_table("bronze_wikidata")

    def get_GMD(self) -> pd.DataFrame:
        """Global Macro Database"""
        return self._fetch_table("bronze_gmd")

    def get_interest(self) -> pd.DataFrame:
        """IMF Zinsdaten"""
        return self._fetch_table("bronze_interest")

    def get_commodity(self) -> pd.DataFrame:
        """IMF Commodity Daten"""
        return self._fetch_table("bronze_commodities")
    
    def get_fx_rates(self) -> pd.DataFrame:
        return self._fetch_table("bronze_fx_rates")
    
    def close(self):
        """Schließt die Datenbankverbindung."""
        self.con.close()

In [4]:
class IMFInteresttoSilver():
    def __init__(self):
        pass
    def split_interest_and_unit(self, df):
        df = df.copy()
        split = df["interest_name"].str.rsplit(",", n=1)
        df["interest_name"] = split.str[0].str.strip()
        df["unit"] = split.str[1].str.strip()
        return df
    def source_naming(self, df):
        df['source'] = 'InternationalMonetaryFund'
        return df
    def affiliation(self, df):
        INTEREST_CATEGORY_MAPPING = {
        # -------------------------
        # Policy Rates
        # -------------------------
        "Monetary policy-related, Rate": "policy_rates",
        "Discount Rate": "policy_rates",
        "Refinancing Rate": "policy_rates",
        "Central bank borrowing facility, Rate": "policy_rates",
        # -------------------------
        # Market Rates
        # -------------------------
        "Money market Rate": "market_rates",
        "Money market rate, Foreign Currency, Rate": "market_rates",
        "Money market rate, maximum, Rate": "market_rates",
        "Money market rate, minimum, Rate": "market_rates",
        "Repurchase agreement Rate": "market_rates",
        "Reverse repurchase agreement Rate": "market_rates",
        "Corporate paper Rate": "market_rates",
        "Certificates of deposit, Rate": "market_rates",
        "Central bank certificates, Rate": "market_rates",
        # -------------------------
        # Deposit Rates
        # -------------------------
        "Deposit Rate": "deposit_rates",
        "Deposit rate, Foreign Currency, Rate": "deposit_rates",
        "Deposit rate, Overnight Rate": "deposit_rates",
        "Deposit rate, Euro, Rate": "deposit_rates",
        "Deposit rate, US dollar, Rate": "deposit_rates",
        "Savings Rate": "deposit_rates",
        "Savings rate, Foreign Currency, Rate": "deposit_rates",
        # Harmonized deposits
        "Harmonized Euro area rates: Deposits, Up to 2 years Outstanding Amounts, Non-financial corporations, Rate": "deposit_rates",
        "Harmonized Euro area rates: Deposits, Up to 1 year New Business, Non-financial corporations, Rate": "deposit_rates",
        "Harmonized Euro area rates: Deposits agreed maturity, Up to 2 years Outstanding Amounts, Households, Rate": "deposit_rates",
        "Harmonized Euro area rates, New business: Deposits agreed maturity, Up to 1 year New Business, Households, Rate": "deposit_rates",
        # -------------------------
        # Lending Rates
        # -------------------------
        "Lending Rate": "lending_rates",
        "Lending rate, Overnight Rate": "lending_rates",
        "Lending rate, Foreign Currency, Rate": "lending_rates",
        "Lending rate, Euro, Rate": "lending_rates",
        "Lending rate, US dollar, Rate": "lending_rates",
        # Harmonized loans
        "Harmonized Euro Area rates: Loans, consumer credit and other, Up to 1 year Outstanding Amounts, Households, Rate": "lending_rates",
        "Harmonized Euro area rates: Loans, New Business, Consumption floating rate, Households, Rate": "lending_rates",
        "Harmonized Euro area rates: Loans, Over 5 years New Business, House purchase, Households, Rate": "lending_rates",
        "Harmonized Euro area rates: Loans, Up to 1 year Outstanding Amounts, Non-financial corporations, Rate": "lending_rates",
        "Harmonized Euro area rates: Loans other than bank overdrafts, Over EUR 1 Million, Over 3 months and up to 1 Year New Business, Non-financial corporations, Rate": "lending_rates",
        "Harmonized Euro area rates, New Business: Loans, House Purchase, households, Over 5 years New Business, House purchase, Households, Rate": "lending_rates",
        # -------------------------
        # Government Rates
        # -------------------------
        "Government bonds, Rate": "government_rates",
        "Government bond yields, minimum, Rate": "government_rates",
        "Government bonds yields, Short to medium term Rate": "government_rates",
        "Government securities: Treasury bills yields, Rate": "government_rates",
        # -------------------------
        # Other / Mixed
        # -------------------------
        "Average cost of funds, Rate": "other_rates",
        "Discount rate, Foreign Currency, Rate": "other_rates",
    }
        df["affiliation"] = (
        df["interest_name"]
        .map(INTEREST_CATEGORY_MAPPING)
        .fillna("other_rates"))
    
        return df
    def add_countryname(self, df):
        df_ = df.rename(columns={"country": "iso3"})
        iso_name = pd.read_csv("gmd.csv", usecols=["countryname", "iso3"])
        iso_name = iso_name.drop_duplicates(subset="iso3")
        print(iso_name.columns)
        print(df_.columns)
        mapping = iso_name.set_index("iso3")["countryname"]
        df_["country"] = df_["iso3"].map(mapping)
        df_ = df_.dropna(subset=["country"])
        return df_


    def run(self,df):
        unit = self.split_interest_and_unit(df)
        source = self.source_naming(unit)
        source['EntityType'] = 'States'
        countrynames = self.add_countryname(source)
        monthly = countrynames[countrynames['frequency'] == "M"]
        return self.affiliation(monthly)
    
df = DataHubReader().get_interest()    
interest = IMFInteresttoSilver().run(df)
interest.to_csv("interest.csv", encoding="utf-8")


Index(['countryname', 'iso3'], dtype='object')
Index(['date', 'iso3', 'indicator', 'interest_name', 'frequency', 'value',
       'ingested_at', 'unit', 'source', 'EntityType'],
      dtype='object')


C:\Users\Konra\AppData\Local\Temp\ipykernel_14400\3909555080.py:77: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df["affiliation"] = (


In [5]:
interest['interest_name'].value_counts()

interest_name
Deposit Rate                                                                                                                                                       5573
Lending Rate                                                                                                                                                       5413
Monetary policy-related, Rate                                                                                                                                      3647
Government securities: Treasury bills yields, Rate                                                                                                                 2893
Money market Rate                                                                                                                                                  2688
Savings Rate                                                                                                                                      

In [6]:
import re
class IMFCommoditytoSilver():
    def __init__(self):
        pass
    def split_commodity_and_unit(self, df):
        df = df.copy()
        def parse_commodity(text):
            if not isinstance(text, str):
                return None, None

            parts = [p.strip() for p in text.split(",") if p.strip()]

            # 1) offensichtliche Metadaten am Ende entfernen
            trailing_noise = {"Unit prices"}
            while parts and (parts[-1] in trailing_noise or re.fullmatch(r"\d{4}=100", parts[-1])):
                parts.pop()

            if not parts:
                return None, None

            # 2) Index-Fälle
            # Sobald irgendwo "Index" oder "Commodity price index" vorkommt:
            # -> unit = "Index"
            # -> commodity_name = erster Teil
            if any(p == "Index" for p in parts) or any("Commodity price index" in p for p in parts):
                return parts[0], "Index"

            # 3) Preis-/Währungs-Einheit suchen
            unit_keywords = [
                "US dollars",
                "US cents",
                "Euro",
                "Thai baht",
                "INR/",
                "USD/",
                "per ",
            ]

            unit_idx = None
            for i, p in enumerate(parts):
                if any(k in p for k in unit_keywords):
                    unit_idx = i
                    break

            if unit_idx is not None:
                commodity_name = ", ".join(parts[:unit_idx]).strip()
                unit = parts[unit_idx].strip()
                return commodity_name, unit

            # 4) Falls keine Einheit erkannt wird:
            # alles als Name behalten, unit = None
            return ", ".join(parts).strip(), None

        parsed = df["commodity_name"].apply(parse_commodity)

        df["commodity_name"] = parsed.str[0]
        df["unit"] = parsed.str[1]

        return df
    def affiliation(self, df):
        commodity_mapping = {
        "Low and Middle Income Commodity Index (World Bank)": "commodity_index",
        "Seafood index": "commodity_index",
        "Agr. Raw Material Index": "commodity_index",
        "Agriculture": "commodity_index",
        "Precious Metals Price Index": "commodity_index",
        "Wool index": "commodity_index",
        "Vegetable oil index": "commodity_index",
        "Timber index": "commodity_index",
        "Sugar index": "commodity_index",
        "Softwood index": "commodity_index",
        "Energy index": "commodity_index",
        "Coffee index": "commodity_index",
        "Coal index": "commodity_index",
        "Cereal index": "commodity_index",
        "Beverages index": "commodity_index",
        "All Metals Index": "commodity_index",
        "All index": "commodity_index",
        "Energy Transition Metal Index": "commodity_index",
        "All Metals EX GOLD Index": "commodity_index",
        "Food and beverage index": "commodity_index",
        "Natural gas index": "commodity_index",
        "Non-Fuel index": "commodity_index",
        "Base Metals index": "commodity_index",
        "Meat Index": "commodity_index",
        "Industrial Materials index": "commodity_index",
        "Hardwood index": "commodity_index",
        "Food index": "commodity_index",
        "Fertlizer": "commodity_index",

        "Propane": "energy",
        "WTI Crude": "energy",
        "Dubai Crude": "energy",
        "Brent Crude": "energy",
        "APSP crude oil($/bbl)": "energy",
        "Natural Gas, US Henry Hub Gas": "energy",
        "LNG, Asia": "energy",
        "Natural gas, EU": "energy",
        "Coal, South Africa": "energy",
        "Coal, Australia": "energy",
        "Uranium": "energy",

        "Rare Earth Elements, Rare earth carbonate REO 42-45 Dom": "metals",
        "Silicon": "metals",
        "Vanadium, Cost, insurance": "metals",
        "Tin": "metals",
        "Nickel": "metals",
        "Copper": "metals",
        "Cobalt": "metals",
        "Chromium, 99.2%, Coarse particle": "metals",
        "Aluminum": "metals",
        "Manganese": "metals",
        "Molybdenum": "metals",
        "Lithium, 99%, Battery Grade": "metals",
        "Lead": "metals",
        "Iron Ore": "metals",
        "Zinc": "metals",

        "Platinum": "precious_metals",
        "Palladium": "precious_metals",
        "Silver": "precious_metals",
        "Gold": "precious_metals",

        "Potassium Fertilizer": "fertilizers",
        "Urea": "fertilizers",
        "Diammonium phosphate": "fertilizers",

        "Soft Sawnwood, Average of Softwoods": "forestry",
        "Hard Sawnwood, Dark Red Meranti": "forestry",
        "Soft Logs": "forestry",
        "Hard Logs, Import Price Japan": "forestry",

        "Fish": "seafood",
        "Shrimp": "seafood",
        "Fish Meal": "seafood",

        "Rice, Thailand": "grains",
        "Oats": "grains",
        "Wheat": "grains",
        "Sorghum": "grains",
        "Barley": "grains",
        "Corn": "grains",

        "Rapeseed Oil": "oils_and_oilseeds",
        "Palm Oil": "oils_and_oilseeds",
        "Olive Oil": "oils_and_oilseeds",
        "Sunflower Oil": "oils_and_oilseeds",
        "Soybeans": "oils_and_oilseeds",
        "Soybeans Oil": "oils_and_oilseeds",
        "Soybean Meal": "oils_and_oilseeds",
        "Groundnuts": "oils_and_oilseeds",
        "Coconut Oil": "oils_and_oilseeds",

        "Poultry": "livestock",
        "Swine": "livestock",
        "Beef": "livestock",
        "Dairy Products, Milk": "livestock",
        "Lamb": "livestock",
        "Hides": "livestock",

        "Rubber": "softs",
        "Orange": "softs",
        "Tea, Colombo": "softs",
        "Tea, Mombasa": "softs",
        "Tea, Kolkata": "softs",
        "Tea, Kenyan": "softs",
        "Cotton": "softs",
        "Coffee, Robustas": "softs",
        "Coffee, Other Mild Arabica": "softs",
        "Cocoa": "softs",
        "Bananas": "softs",
        "Wool, Fine": "softs",
        "Wool, Coarse": "softs",

        "Vegetables, Tomato": "food",
        "Non-Citrus Fruit, Apple": "food",
        "Legumes, Chickpea": "food",
    }

        df = df.copy()
        df["affiliation"] = df["commodity_name"].map(commodity_mapping).fillna("other_commodities")
        return df
    def source_naming(self, df):
        df['soruce'] = 'InternationalMonetaryFund'
        return df
    def run(self, df):
        unit = self.split_commodity_and_unit(df)
        affiliation = self.affiliation(unit)
        source = self.source_naming(affiliation)
        source['EntityType'] = 'Commodity'
        monthly = source[source['frequency'] == "M"]
        return monthly

commo = DataHubReader().get_commodity()
commodity = IMFCommoditytoSilver().run(commo)
commodity.to_csv("commodity.csv", encoding="utf-8")